In [1]:
"""
- This module contain the code for the full attack to work.
- You don't need to install or load anything,
- You need only a working sagemath server.
- Online testing : https://sagecell.sagemath.org/
"""


def coppersmith_howgrave_univariate(pol, modulus, beta, mm, tt, XX):
    """
    Coppersmith revisited by Seck et al.

    finds a solution if:
    * b|modulus, b >= modulus^beta , 0 < beta <= 1
    * |x| < XX

    # Copyright : Michel Seck
    # Github : https://github.com/Fayeabdoulaye/generalized_wiener_type_attacks

    """
    #
    # init
    #
    dd = pol.degree()
    nn = dd * mm + tt

    #
    # checks
    #
    if not 0 < beta <= 1:
        raise ValueError("beta should belongs in (0, 1]")

    if not pol.is_monic():
        raise ArithmeticError("Polynomial must be monic.")

    # Coppersmith revisited algo for univariate
    # change ring of pol and x
    polZ = pol.change_ring(ZZ)
    x = polZ.parent().gen()
    # compute polynomials
    gg = []
    for ii in range(mm):
        for jj in range(dd):
            gg.append((x * XX)**jj * modulus**(mm - ii) * polZ(x * XX)**ii)
    for ii in range(tt):
        gg.append((x * XX)**ii * polZ(x * XX)**mm)

    # construct lattice B
    BB = Matrix(ZZ, nn)
    for ii in range(nn):
        for jj in range(ii+1):
            BB[ii, jj] = gg[ii][jj]
    # LLL
    BB = BB.LLL()
    # transform shortest vector in polynomial
    new_pol = 0
    for ii in range(nn):
        new_pol += x**ii * BB[0, ii] / XX**ii
    # factor polynomial
    potential_roots = new_pol.roots()
    # test roots
    roots = []
    for root in potential_roots:
        if root[0].is_integer():
            result = polZ(ZZ(root[0]))
            if gcd(modulus, result) >= modulus^beta:
                roots.append(ZZ(root[0]))
    return roots


def get_p(N, pa):
    """Returns a factor p of N or None given N = pq and an approximation of p."""
    F.<x> = PolynomialRing(Zmod(N), implementation='NTL');
    pol = x - pa
    dd = pol.degree()
    beta = 0.5                             # we should have q >= N^beta
    epsilon = beta / 7                     # <= beta/7
    mm = ceil(beta**2 / (dd * epsilon))    # optimized
    tt = floor(dd * mm * ((1/beta) - 1))   # optimized
    XX = ceil(N**((beta**2/dd) - epsilon)) # we should have |diff| < X

    # Coppersmith

    roots = coppersmith_howgrave_univariate(pol, N, beta, mm, tt, XX)
    if roots:
        return pa - roots[0]


def rand_primes(size, mu):
    """Generates random primes with q < p < mu*q."""
    p = random_prime(1 << (size - 1), 1 << size)
    while True:
        q = random_prime(1 << (size - 1), 1 << size)
        if p < q:
            p, q = q, p
            if q < p < mu*q:
                break

    return p, q


def get_approx_p_pm_q(N, e, x, y, prec=1000):
    """Returns an approximate p+q and p-q given N, e, x, y such that

        ex - (p - 1)^2(q - 1)^2y = w and N=pq
    """
    RF = RealField(prec)
    N, e, x, y = RF(N), RF(e), RF(x), RF(y)
    inner_sqrt = sqrt(e * x / y)
    p_plus_q = floor(N + 1 - inner_sqrt)
    disc = (p_plus_q)**2 - 4 * N
    if disc < 0:
        return None, None
    p_minus_q = floor(sqrt(disc))
    return p_plus_q, p_minus_q




def gen_weak_RSA_instance(nbits, mu, prec=1024):
    """Generates weak public key instances of the RSA-like cryptosystem from
       Seck et al. (AfricaCrypt 2025)
    """
    RF = RealField(prec)
    p, q = rand_primes(nbits//2, mu); N = p*q; phi = (p-1)**2*(q-1)**2
    N_squared = RF(N**2); N_ss = RF(N**4)
    threshold = (2*mu*N_squared - (mu - 1)**2*N + 2*mu) / ((2*(3*mu**2 + 4*mu**(3/2) + 4*mu + 1)*N) + 8*mu*(mu**(1/2) + 1)*RF(N**(3/2)) + (4*(mu**(1/2) - 1) + 8*mu*(mu**(1/2) + 1))*RF(N**(1/2))
    )
    found = False
    while not found:
        d = randint(1, round(sqrt(threshold)) - 1)
        if gcd(d, phi) == 1:
            found = True
    e = inverse_mod(phi-d, phi)
    return N, e


def factor_N(N, e, mu, prec=None, bug=False):
    """Returns the factors p, q of N or None given the public key (N, e)."""

    i = x = y = p_plus_q = p_minus_q = pa = p = q = None

    try:
        precision = prec if prec else 3 * N.bit_length()

        r = 2 * mu * e / (2 * mu * N**2 - (mu - 1)**2 * N + 2 * mu)
        cf = r.continued_fraction()

        print(f"Continued Fraction cf: {cf}")

        for i, xy in enumerate(cf.convergents()[1:]):
            x = xy.denominator()
            y = xy.numerator()

            p_plus_q, p_minus_q = get_approx_p_pm_q(
                N, e, x, y, precision
            )

            pa = (p_plus_q + p_minus_q) // 2

            p = get_p(N, pa)

            if p:
                q = N // p
                return (i, x, y, p_plus_q, p_minus_q, pa, p, q)

        return None, None

    except Exception:
        if bug:
            raise

        return (i, x, y, p_plus_q, p_minus_q, pa, p, q)

# A test
nbits_modulus = 200; mu = 2
print("Generation of a weak RSA instance (N, e)")
N = 265839721739624484719433732023423940832894821073878576476773
e = 11088003240518255031282718983499356127491698185331164614144898087286354771002542344519459394576885438741530757179984727
#N, e = gen_weak_RSA_instance(nbits_modulus, mu);
print(f"{N=} \n{40*'='}\n{e=}\n")

res = factor_N(N, e, mu, bug=False)
i, x, y, p_plus_q, p_minus_q, pa, p, q = res
print(f"Convergent {i}:\n x = {x}\n y = {y}")
print(f"Approximations: \n p_plus_q = {p_plus_q} \n p_minus_q = {p_minus_q}")
print(f"pa = {pa}")
print(f"Get \np={p} \n and \nq={q}")


Generation of a weak RSA instance (N, e)
N=265839721739624484719433732023423940832894821073878576476773 
e=11088003240518255031282718983499356127491698185331164614144898087286354771002542344519459394576885438741530757179984727

Continued Fraction cf: [0; 6, 2, 1, 2, 10, 1, 40, 1, 1, 1, 2, 1, 1, 5, 20, 1, 1, 1, 1, 5, 7, 4, 1, 1, 1, 1, 1, 5, 3, 1, 8, 117, 6, 2, 8, 1, 1, 4, 5, 2, 2, 3, 1, 1, 2, 1, 1, 1, 4, 1, 4, 1, 2, 2, 1, 1, 1, 3, 1, 1, 2, 2, 2, 2, 9, 2, 12, 26, 6, 18, 1, 1, 4, 1, 2, 1, 1, 1, 7, 33, 2, 1, 1, 1, 4, 1, 369, 4, 2, 2, 2, 1, 2, 14, 1, 2, 3, 1, 1, 2, 1, 1, 3, 2, 9, 2, 1, 2, 7, 1, 4, 6, 1, 3, 7, 1, 1, 9, 1, 20, 1, 936, 3, 8, 3, 1, 1, 3, 1, 5, 2, 18, 2, 1, 11, 3, 225, 2, 4, 33, 1, 32, 1, 1, 3, 1, 2, 8, 2, 2, 1, 2, 3, 1, 4, 2, 1, 2, 1, 2, 3, 7, 3, 10, 5, 1, 1, 1, 2, 1, 1, 1, 13, 9, 2, 3, 1, 3, 2, 11, 1, 1, 1, 1, 1, 28, 2, 10, 1, 8, 8, 5, 5, 11, 3, 1, 3, 46, 1, 1, 2, 1, 7, 3, 8, 2, 1, 1, 2, 8, 32, 5, 1, 26, 1, 43, 1, 3, 1, 2, 3, 36, 2, 1, 1, 27, 1, 2, 2]
Convergent 30:
 x = 83688